In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the CloudGPU/python Docker image: https://github.com/CloudGPU/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/workspace/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/workspace/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /CloudGPU/temp/, but they won't be saved outside of the current session

In [1]:
import subprocess, os, re

if not os.path.exists("/workspace/working/ComfyUI"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/comfyanonymous/ComfyUI.git",
                    "/workspace/working/ComfyUI"], check=True)
os.chdir("/workspace/working/ComfyUI")

# 精确跳过 torch/torchvision/torchaudio，不误杀 torchsde
def pkg_name(line):
    return re.split(r'[>=<!;\s]', line.strip())[0].lower()

skip = {"torch", "torchvision", "torchaudio"}
with open("requirements.txt") as f:
    pkgs = [l.strip() for l in f
            if l.strip() and not l.startswith("#") and not l.startswith("-e")
            and l.strip() != "." and pkg_name(l) not in skip]
subprocess.run(["pip", "install", "-q"] + pkgs, check=False)
subprocess.run(["pip", "install", "-q", "torchsde"], check=True)  # 确保装上

nodes = [
    "https://github.com/ltdrdata/ComfyUI-Manager",
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus",
    "https://github.com/Fannovel16/comfyui_controlnet_aux",
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/WASasquatch/was-node-suite-comfyui",
]
for url in nodes:
    name = url.split("/")[-1]
    dest = f"custom_nodes/{name}"
    if not os.path.exists(dest):
        r = subprocess.run(["git", "clone", "--depth", "1", url, dest])
        if r.returncode != 0:
            print(f"⚠️ {name} 克隆失败，跳过")

subprocess.run(["pip", "install", "-q", "insightface", "onnxruntime-gpu"])
print("✅ Cell 1 完成")

Cloning into '/workspace/working/ComfyUI'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.8/90.8 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.6/794.6 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.8/92.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.8/320.8 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/

Cloning into 'custom_nodes/ComfyUI-Manager'...
Cloning into 'custom_nodes/ComfyUI_IPAdapter_plus'...
Cloning into 'custom_nodes/comfyui_controlnet_aux'...
Cloning into 'custom_nodes/ComfyUI-KJNodes'...
Cloning into 'custom_nodes/was-node-suite-comfyui'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 7.4 MB/s eta 0:00:00
✅ Cell 1 完成


In [2]:
import os
import glob

print("正在配置 CLIP Vision 模型...")

# 1. 确保 ComfyUI 的目标文件夹存在
target_dir = '/workspace/working/ComfyUI/models/clip_vision/'
os.makedirs(target_dir, exist_ok=True)

# 2. 自动在 /workspace/input/ 目录下地毯式搜索刚刚挂载的模型
source_files = glob.glob('/workspace/input/**/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors', recursive=True)

if source_files:
    source_path = source_files[0]
    target_path = os.path.join(target_dir, 'CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors')
    
    # 3. 建立软链接 (如果之前有错误的残留，先删除)
    !rm -f {target_path}
    !ln -s {source_path} {target_path}
    
    print(f"✅ 模型挂载成功！")
    print(f"源文件: {source_path}")
    print(f"已链接至: {target_path}")
else:
    print("❌ 错误：未在 /workspace/input/ 中找到模型。")
    print("请检查右侧面板的 'Input' 中是否已成功 Add 了刚才下载的 Notebook Output。")

正在配置 CLIP Vision 模型...
✅ 模型挂载成功！
源文件: /workspace/input/notebooks/yang084/download-clip-vision-h/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors
已链接至: /workspace/working/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors


In [3]:
# 本地/云端端口开放与访问服务启动
# 配置云端实例对外开放端口的环境（具体请参考对应服务商手册）。
# 部署时可依据自身的云服务器环境配置对应的防火墙及端口映射或使用标准SSH隧道访问。

In [4]:
import yaml, os

os.chdir("/workspace/working/ComfyUI")

extra_paths = {
    "comfyui_models": {
        "base_path":      "/workspace/input/datasets/yang084/comfyui-models/comfyui-models/",
        "checkpoints":    "checkpoints/",
        "ipadapter":      "ipadapter/",
        "clip_vision":    "clip_vision/",
        "upscale_models": "upscale_models/",
        "insightface":    "insightface/",
    },
    "comfyui_loras": {
        "base_path": "/workspace/input/datasets/yang084/comfyui-loras/",
        "loras":     "loras/",
    },
}
with open("extra_model_paths.yaml", "w") as f:
    yaml.dump(extra_paths, f)

for name, path in {
    "checkpoints": "/workspace/input/datasets/yang084/comfyui-models/comfyui-models/checkpoints",
    "loras":       "/workspace/input/datasets/yang084/comfyui-loras/loras",
}.items():
    print(f"{'✅' if os.path.exists(path) else '⚠️'} {name}: {path}")
print("✅ Cell 3 完成")

✅ checkpoints: /workspace/input/datasets/yang084/comfyui-models/comfyui-models/checkpoints
✅ loras: /workspace/input/datasets/yang084/comfyui-loras/loras
✅ Cell 3 完成


In [5]:
# 1. 确保目标文件夹（两层 models）被创建出来
!mkdir -p /workspace/working/ComfyUI/models/insightface/models/

# 2. 把只读数据集里的 antelopev2 整个文件夹强制拷贝过去（覆盖可能存在的空文件夹）
!cp -r /workspace/input/datasets/yang084/comfyui-models/comfyui-models/insightface/antelopev2 /workspace/working/ComfyUI/models/insightface/models/

# 3. 顺手检查一下有没有拷成功
!ls /workspace/working/ComfyUI/models/insightface/models/antelopev2/

1k3d68.onnx    genderage.onnx  scrfd_10g_bnkps.onnx
2d106det.onnx  glintr100.onnx


In [6]:
# 1. 进入 CloudGPU 的可写区自定义节点目录
%cd /workspace/working/ComfyUI/custom_nodes/

# 2. 克隆 ComfyUI-Essentials 插件包
!git clone https://github.com/cubiq/ComfyUI_essentials

# 3. 检查是否克隆成功
!ls /workspace/working/ComfyUI/custom_nodes/

/workspace/working/ComfyUI/custom_nodes
Cloning into 'ComfyUI_essentials'...
remote: Enumerating objects: 480, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 480 (delta 125), reused 114 (delta 114), pack-reused 331 (from 2)
Receiving objects: 100% (480/480), 166.13 KiB | 3.02 MiB/s, done.
Resolving deltas: 100% (287/287), done.
comfyui_controlnet_aux	ComfyUI-KJNodes		 was-node-suite-comfyui
ComfyUI_essentials	ComfyUI-Manager		 websocket_image_save.py
ComfyUI_IPAdapter_plus	example_node.py.example


In [7]:
# 假设你的 ComfyUI 安装在 /workspace/working/ComfyUI
%cd /workspace/working/ComfyUI/custom_nodes

# 1. WAS Node Suite (包含强大的图像处理、建立纯色画布、添加文字功能)
!git clone https://github.com/WASasquatch/was-node-suite-comfyui

# 2. Rembg ComfyUI Node (用于一键去除原图的灰色/杂色背景，提取透明人物)
!git clone https://github.com/Jcd1230/rembg-comfyui-node

# 3. ComfyUI Essentials (包含精确的图像缩放 Image Resize、平移和合成 Image Composite)
!git clone https://github.com/cubiq/ComfyUI_essentials

# 回到主目录并安装缺失的 Python 依赖包
%cd /workspace/working/ComfyUI
!pip install -r custom_nodes/was-node-suite-comfyui/requirements.txt
!pip install rembg

/workspace/working/ComfyUI/custom_nodes
fatal: destination path 'was-node-suite-comfyui' already exists and is not an empty directory.
Cloning into 'rembg-comfyui-node'...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 7 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (7/7), done.
fatal: destination path 'ComfyUI_essentials' already exists and is not an empty directory.
/workspace/working/ComfyUI
  Cloning https://github.com/WASasquatch/img2texture.git to /tmp/pip-req-build-z_qfn2st
  Running command git clone --filter=blob:none --quiet https://github.com/WASasquatch/img2texture.git /tmp/pip-req-build-z_qfn2st
  Resolved https://github.com/WASasquatch/img2texture.git to commit d6159abea44a0b2cf77454d3d46962c8b21eb9d3
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/WASasquatch/cstr to /tmp/pip-req-build-x2xrmydl
  Running command git clone -

In [8]:
# 1. 回到 ComfyUI 主目录
%cd /workspace/working/ComfyUI

# 2. 创建存放 inpaint 模型的专属文件夹（如果没有的话）
!mkdir -p models/inpaint

# 3. 使用 wget 直接从官方源下载大号 Lama 模型 (big-lama.pt)
!wget -O models/inpaint/big-lama.pt https://github.com/Sanster/models/releases/download/add_big_lama/big-lama.pt

print("✅ Lama 模型下载完成！请回到网页刷新！")

/workspace/working/ComfyUI
--2026-04-13 14:43:24--  https://github.com/Sanster/models/releases/download/add_big_lama/big-lama.pt
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/143410310/22b2930e-5328-4ff1-8537-46332eca8550?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-13T15%3A16%3A27Z&rscd=attachment%3B+filename%3Dbig-lama.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-13T14%3A15%3A28Z&ske=2026-04-13T15%3A16%3A27Z&sks=b&skv=2018-11-09&sig=wQRxL%2BuWPZUAXNsvdQ%2BTdt2EROUBY7BQX8omvclfm7w%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NjA5NTAwNCwibmJmIjoxNzc2MDkxNDA0LCJwYXRoIjoicmVsZWFzZW

In [9]:
# 1. 进入 custom_nodes 目录
%cd /workspace/working/ComfyUI/custom_nodes

# 2. 克隆目前最好用的 Inpaint 插件包（包含 Lama）
!git clone https://github.com/Acly/comfyui-inpaint-nodes

# 3. 安装该插件需要的 Python 依赖
%cd /workspace/working/ComfyUI/custom_nodes/comfyui-inpaint-nodes
!pip install -r requirements.txt

# 4. 回到主目录
%cd /workspace/working/ComfyUI

/workspace/working/ComfyUI/custom_nodes
Cloning into 'comfyui-inpaint-nodes'...
remote: Enumerating objects: 300, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 300 (delta 99), reused 84 (delta 73), pack-reused 185 (from 1)
Receiving objects: 100% (300/300), 3.12 MiB | 19.14 MiB/s, done.
Resolving deltas: 100% (186/186), done.
/workspace/working/ComfyUI/custom_nodes/comfyui-inpaint-nodes
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
/workspace/working/ComfyUI


In [10]:
import subprocess, os

os.chdir("/workspace/working/ComfyUI")

with open("/workspace/working/comfyui.log", "w") as log:
    subprocess.Popen(
        ["python", "main.py",
         "--listen", "0.0.0.0", "--port", "8188",
         "--extra-model-paths-config", "extra_model_paths.yaml"],
        stdout=log, stderr=subprocess.STDOUT,
        preexec_fn=os.setsid, close_fds=True
    )
print("✅ Cell 4 完成：ComfyUI 在后台启动")

✅ Cell 4 完成：ComfyUI 在后台启动


In [ ]:
# 本地/云端端口开放与访问服务启动
# 配置云端实例对外开放端口的环境（具体请参考对应服务商手册）。
# 部署时可依据自身的云服务器环境配置对应的防火墙及端口映射或使用标准SSH隧道访问。